<a href="https://colab.research.google.com/github/Edomario082909/AHHHHR/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# 1. Preparazione Ambiente e Performance
!apt -y update -qq
!apt -y install -qq aria2
!wget https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4

# 2. Installazione ComfyUI Core
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# 3. Installazione CUSTOM NODES (Per risolvere i nodi rossi)
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/kijai/ComfyUI-WanVideo.git
!git clone https://github.com/city96/ComfyUI-GGUF.git
%cd /content/ComfyUI

# 4. Installazione Dipendenze aggiornate (Torch 2.5+ per WanVideo)
!pip install -q xformers torchsde omegaconf diffusers accelerate
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-WanVideo/requirements.txt

# 5. Download MODELLI Wan2.1 (Versione ottimizzata)
# Scarichiamo il modello T2V 1.3B (più veloce su Colab) e il VAE
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B/resolve/main/diffusion_pytorch_model.safetensors" -d /content/ComfyUI/models/checkpoints -o wan2.1_t2v_1.3b.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B/resolve/main/models_vae_diffusion_pytorch_model.safetensors" -d /content/ComfyUI/models/vae -o wan2.1_vae.safetensors

# 6. Avvio con Cloudflare Tunnel
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64
import atexit, requests, subprocess, time, re, os
from random import randint
from threading import Timer
from queue import Queue

def cloudflared(port, metrics_port, output_queue):
    atexit.register(lambda p: p.terminate(), subprocess.Popen(['/content/cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
    attempts, tunnel_url = 0, None
    while attempts < 10 and not tunnel_url:
        attempts += 1
        time.sleep(3)
        try:
            tunnel_url = re.search("(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text).group("url")
        except:
            pass
    if not tunnel_url:
        raise Exception("Can't connect to Cloudflare Edge")
    output_queue.put(tunnel_url)

output_queue, metrics_port = Queue(), randint(8100, 9000)
thread = Timer(2, cloudflared, args=(8188, metrics_port, output_queue))
thread.start()
thread.join()
tunnel_url = output_queue.get()
print(f"\n\nIL TUO LINK COMFYUI: {tunnel_url}\n\n")

!python main.py --dont-print-server